<a href="https://colab.research.google.com/github/chight1/PaddlePaddle-Practice/blob/main/HousingPrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install paddlepaddle
import pandas as pd
from sklearn.preprocessing import StandardScaler
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
features_to_drop = ['Alley', 'FireplaceQu', 'PoolQC', 'MiscFeature', 'Fence', 'MasVnrType', 'LotFrontage', 'GarageQual', 'GarageCond', 'BsmtCond']
train_df = train_df.drop(features_to_drop, axis=1)
test_df = test_df.drop(features_to_drop, axis=1)

cat_features = ['BsmtQual', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish']
train_df[cat_features] = train_df[cat_features].fillna('NA')
test_df[cat_features] = test_df[cat_features].fillna('NA')

median_masvnr = train_df['MasVnrArea'].median()
median_garageyr = train_df['GarageYrBlt'].median()
train_df['MasVnrArea'] = train_df['MasVnrArea'].fillna(median_masvnr)
train_df['GarageYrBlt'] = train_df['GarageYrBlt'].fillna(median_garageyr)
test_df['MasVnrArea'] = test_df['MasVnrArea'].fillna(median_masvnr)
test_df['GarageYrBlt'] = test_df['GarageYrBlt'].fillna(median_garageyr)

mode_electrical = train_df['Electrical'].mode()[0]
train_df['Electrical'] = train_df['Electrical'].fillna(mode_electrical) ## only one null
test_df['Electrical'] = test_df['Electrical'].fillna(mode_electrical)

for col in ['MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']:
    test_df[col] = test_df[col].fillna(train_df[col].mode()[0])

for col in ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
            'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']:
    test_df[col] = test_df[col].fillna(train_df[col].median())

# get null counts of each column
null_counts = test_df.isnull().sum()
null_counts[null_counts > 0]
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
scaler = StandardScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
print("Features before:", train_df.shape[1])
train_df = pd.get_dummies(train_df)
print("Features after:", train_df.shape[1])
# correlation matrix
corr_matrix = train_df.corr().abs()

# Keep features with correlation >= 0.3 to SalePrice
strong_corr = corr_matrix['SalePrice'][corr_matrix['SalePrice'] >= 0.4].index

# Drop all weak corr features
train_df_reduced = train_df[strong_corr]

print("Original features:", train_df.shape[1])
print("Reduced features:", train_df_reduced.shape[1])

Features before: 71
Features after: 259
Original features: 259
Reduced features: 27


In [9]:
X_train = train_df_reduced.drop('SalePrice', axis=1)
y_train = train_df_reduced['SalePrice']

In [10]:
print(X_train.dtypes[X_train.dtypes == 'object'])

Series([], dtype: object)


In [17]:
import paddle
from paddle.nn import Linear
import paddle.nn.functional as F

paddle.set_device('cpu')

model = Linear(in_features=X_train.shape[1], out_features=1)

# Convert DFs to paddle tensors
X_train = X_train.astype(float)
y_train = y_train.astype(float)
X_tensor = paddle.to_tensor(X_train.values, dtype='float32')
y_tensor = paddle.to_tensor(y_train.values.reshape(-1, 1), dtype='float32')

# optimizer and loss function
optimizer = paddle.optimizer.Adam(learning_rate=0.01, parameters=model.parameters())
loss_fn = paddle.nn.MSELoss()

epochs = 200

for epoch in range(epochs):
    # Forward pass
    y_pred = model(X_tensor)

    # Compute loss
    loss = loss_fn(y_pred, y_tensor)

    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    optimizer.clear_grad()

    # Log training progress every 100 epochs
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.numpy()}')

Epoch [10/200], Loss: 1.1095470190048218
Epoch [20/200], Loss: 0.8411771655082703
Epoch [30/200], Loss: 0.6288743019104004
Epoch [40/200], Loss: 0.4491651654243469
Epoch [50/200], Loss: 0.35930266976356506
Epoch [60/200], Loss: 0.29266247153282166
Epoch [70/200], Loss: 0.2566882073879242
Epoch [80/200], Loss: 0.23369747400283813
Epoch [90/200], Loss: 0.2199242115020752
Epoch [100/200], Loss: 0.21076582372188568
Epoch [110/200], Loss: 0.2043948918581009
Epoch [120/200], Loss: 0.19969668984413147
Epoch [130/200], Loss: 0.19606849551200867
Epoch [140/200], Loss: 0.19320383667945862
Epoch [150/200], Loss: 0.19091267883777618
Epoch [160/200], Loss: 0.18907533586025238
Epoch [170/200], Loss: 0.18760305643081665
Epoch [180/200], Loss: 0.18642523884773254
Epoch [190/200], Loss: 0.18548469245433807
Epoch [200/200], Loss: 0.1847350299358368
